# 03. Sprawdzenie danych testowych

Notebook odtwarza dane testowe z rozdziałów 11.1 do 11.3 dokumentu [`metody-statystyczne.md`](../metody-statystyczne.md) za pomocą modułu [`stats_methods.py`](stats_methods.py), z którego korzysta notebook `02-descriptive-stats.ipynb`. Pokazuje, że liczby w dokumencie są poprawne i że moduł liczy zgodnie z wariantami opisanymi w dokumencie.

Wartości w dokumencie są zaokrąglone do 10 miejsc po przecinku, dlatego porównanie dopuszcza różnicę do 0,55 jednostki na 10. miejscu.

In [1]:
import math
import sys
import unittest

import pandas as pd
from scipy import stats

import stats_methods as sm

TOLERANCE = 5.5e-11

# Chapter 11.1: five prices on consecutive days.
prices = pd.Series([100, 110, 99, 108.9, 102.366], index=pd.date_range("2026-01-01", periods=5, freq="D"))
simple = sm.returns(prices, "simple")
log = sm.returns(prices, "log")
desc = sm.descriptive(prices)
vol = sm.volatility(prices)
shape = sm.shape(prices)
_, dd = sm.drawdown(prices)

# Chapter 11.2: two return series entered directly.
days = pd.date_range("2026-01-02", periods=4, freq="D")
market = sm.correlation_beta(pd.Series([0.10, -0.10, 0.10, -0.06], index=days),
                             pd.Series([0.02, -0.01, 0.03, -0.02], index=days))


# Chapter 11.3: special cases.
def dated(pairs):
    return pd.Series([v for _, v in pairs], index=pd.to_datetime([d for d, _ in pairs]), dtype=float)


def consecutive(values):
    return pd.Series(values, index=pd.date_range("2026-01-01", periods=len(values), freq="D"), dtype=float)


zero = sm.descriptive(consecutive([100, 110, 110, 99]))
gap_prices = dated([("2026-01-01", 100), ("2026-01-02", 110), ("2026-01-04", 121)])
gap = sm.descriptive(gap_prices)
ema_prices = dated([("2026-01-01", 10), ("2026-01-02", 12), ("2026-01-03", 14),
                    ("2026-01-05", 20), ("2026-01-06", 22), ("2026-01-07", 25)])
ema_gap = sm.ema(ema_prices, 2)
missing = sm.descriptive(consecutive([100, float("nan"), 110, 121, 133.1]))
infinite = sm.descriptive(consecutive([100, float("inf"), 110, 121, 133.1]))
constant_prices = consecutive([100] * 5)
constant = sm.descriptive(constant_prices)
constant_shape = sm.shape(constant_prices)
_, constant_dd = sm.drawdown(constant_prices)
empty = sm.descriptive(pd.Series([], index=pd.DatetimeIndex([]), dtype=float))

## 1. Porównanie z wartościami z dokumentu

In [2]:
checks = [
    ("11.1", "zwroty proste", list(simple.round(10)), [0.10, -0.10, 0.10, -0.06]),
    ("11.1", "zwrot logarytmiczny, dzień 1", log.iloc[0], 0.0953101798),
    ("11.1", "zwrot logarytmiczny, dzień 2", log.iloc[1], -0.1053605157),
    ("11.1", "zwrot logarytmiczny, dzień 3", log.iloc[2], 0.0953101798),
    ("11.1", "zwrot logarytmiczny, dzień 4", log.iloc[3], -0.0618754037),
    ("11.1", "suma zwrotów logarytmicznych", log.sum(), 0.0233844402),
    ("11.1", "średnia zwrotów prostych", desc["mean"], 0.01),
    ("11.1", "mediana zwrotów prostych", desc["median"], 0.02),
    ("11.1", "odchylenie zwrotów log., n−1", vol["daily"], 0.1048185044),
    ("11.1", "odchylenie razy √365", vol["annual"], 2.0025547144),
    ("11.1", "zwrot za okres", desc["total_return"], 0.02366),
    ("11.1", "maksymalne obsunięcie", dd["max_drawdown"], -0.10),
    ("11.1", "cena w szczycie", prices[pd.Timestamp(dd["peak_date"])], 110),
    ("11.1", "cena w dołku", prices[pd.Timestamp(dd["trough_date"])], 99),
    ("11.1", "powrót do szczytu", dd["recovery_date"], None),
    ("11.1", "SMA z 3 dni na końcu", sm.sma(prices, 3).iloc[-1], 103.422),
    ("11.1", "skośność G1 zwrotów log.", shape["skewness"], -0.1468980258),
    ("11.1", "kurtoza G2 zwrotów log.", shape["kurtosis"], -5.1579652449),
    ("11.2", "korelacja", market["pearson"], 0.9317610191),
    ("11.2", "beta", market["beta"], 4.1176470588),
    ("11.3", "zerowy zwrot: liczba zwrotów", zero["returns"], 3),
    ("11.3", "zerowy zwrot: dni bez zmiany", zero["zero_days"], 1),
    ("11.3", "zerowy zwrot: odsetek dni wzrostowych", zero["up_share"], 0.5),
    ("11.3", "luka: liczba poprawnych zwrotów", gap["returns"], 1),
    ("11.3", "luka: średnia pusta", math.isnan(gap["mean"]), True),
    ("11.3", "luka: zwrot za okres", gap["total_return"], 0.21),
    ("11.3", "luka: dni kalendarzowe", gap["calendar_days"], 3),
    ("11.3", "luka: SMA z 3 dni 4 stycznia pusta", math.isnan(sm.sma(gap_prices, 3).iloc[-1]), True),
    ("11.3", "luka: EMA z 3 dni 4 stycznia pusta", math.isnan(sm.ema(gap_prices, 3).iloc[-1]), True),
    ("11.3", "EMA po luce, 2 stycznia", ema_gap.loc["2026-01-02"], 11),
    ("11.3", "EMA po luce, 3 stycznia", ema_gap.loc["2026-01-03"], 13),
    ("11.3", "EMA po luce, 5 stycznia pusta", math.isnan(ema_gap.loc["2026-01-05"]), True),
    ("11.3", "EMA po luce, 6 stycznia", ema_gap.loc["2026-01-06"], 21),
    ("11.3", "EMA po luce, 7 stycznia", ema_gap.loc["2026-01-07"], 23.6666666667),
    ("11.3", "brakująca cena: liczba cen", missing["prices"], 4),
    ("11.3", "brakująca cena: liczba zwrotów", missing["returns"], 2),
    ("11.3", "cena nieskończona: liczba zwrotów", infinite["returns"], 2),
    ("11.3", "stała cena: średnia", constant["mean"], 0),
    ("11.3", "stała cena: odchylenie", sm.volatility(constant_prices)["daily"], 0),
    ("11.3", "stała cena: odsetek dni wzrostowych pusty", math.isnan(constant["up_share"]), True),
    ("11.3", "stała cena: skośność pusta", math.isnan(constant_shape["skewness"]), True),
    ("11.3", "stała cena: kurtoza pusta", math.isnan(constant_shape["kurtosis"]), True),
    ("11.3", "stała cena: maksymalne obsunięcie", constant_dd["max_drawdown"], 0),
    ("11.3", "pusta próba: przyczyna podana", isinstance(empty["reason"], str), True),
]


def matches(value, expected):
    if expected is None or isinstance(expected, (list, bool)):
        return value == expected
    return abs(value - expected) <= TOLERANCE


result = pd.DataFrame(
    [(chapter, name, value, expected, matches(value, expected)) for chapter, name, value, expected in checks],
    columns=["rozdział", "wartość", "obliczona", "w dokumencie", "zgodna"],
)
pd.set_option("display.precision", 10)
result

,rozdział,wartość,obliczona,w dokumencie,zgodna
0,11.1,zwroty proste,"[0.1, -0.1, 0.1, -0.06]","[0.1, -0.1, 0.1, -0.06]",True
1,11.1,"zwrot logarytmiczny, dzień 1",0.0953101798,0.0953101798,True
2,11.1,"zwrot logarytmiczny, dzień 2",-0.1053605157,-0.1053605157,True
3,11.1,"zwrot logarytmiczny, dzień 3",0.0953101798,0.0953101798,True
4,11.1,"zwrot logarytmiczny, dzień 4",-0.0618754037,-0.0618754037,True
5,11.1,suma zwrotów logarytmicznych,0.0233844402,0.0233844402,True
6,11.1,średnia zwrotów prostych,0.01,0.01,True
7,11.1,mediana zwrotów prostych,0.02,0.02,True
8,11.1,"odchylenie zwrotów log., n−1",0.1048185044,0.1048185044,True
9,11.1,odchylenie razy √365,2.0025547144,2.0025547144,True


## 2. Zgodność z bibliotekami

Dokument wskazuje skorygowaną skośność i kurtozę, jak `SKEW` i `KURT` w Excelu. W `scipy` odpowiada im wywołanie z parametrem `bias=False`; domyślne wywołanie daje wersję nieskorygowaną i inny wynik, przed czym ostrzega rozdział 8.1.

In [3]:
pd.DataFrame({
    "stats_methods": [shape["skewness"], shape["kurtosis"]],
    "scipy, bias=False": [stats.skew(log, bias=False), stats.kurtosis(log, bias=False)],
    "scipy, domyślnie": [stats.skew(log), stats.kurtosis(log)],
}, index=["skośność", "kurtoza"])

,stats_methods,"scipy, bias=False","scipy, domyślnie"
skośność,-0.1468980258,-0.1468980258,-0.0848116148
kurtoza,-5.1579652449,-5.1579652449,-1.8877286993


In [4]:
assert result["zgodna"].all(), result[~result["zgodna"]]
print(f"Wszystkie {len(result)} wartości z rozdziału 11 są zgodne z dokumentem.")

Wszystkie 44 wartości z rozdziału 11 są zgodne z dokumentem.


## 3. Testy automatyczne

Plik [`test_stats_methods.py`](test_stats_methods.py) sprawdza te same przypadki oraz walidację danych na syntetycznych świecach, bez połączenia z API: brakujące i nieskończone wartości, świecę o złej godzinie otwarcia, braki na początku i końcu okresu, odrzucenie wszystkich rekordów, duplikaty i nienumeryczną wartość w odpowiedzi API. Testy można uruchomić także bez notebooka poleceniem `python -m unittest -v test_stats_methods` w tym katalogu.

In [5]:
import test_stats_methods

suite = unittest.defaultTestLoader.loadTestsFromModule(test_stats_methods)
outcome = unittest.TextTestRunner(stream=sys.stdout, verbosity=1).run(suite)
assert outcome.wasSuccessful()

.

.

.

.

.

.

... (16 outputs left out)

.

.

.

----------------------------------------------------------------------
Ran 25 tests in 0.140s

OK
